Semantic Caching

In [ ]:
from datetime import datetime
import uuid
import os
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
from EmbeddingFunction import MyEmbeddingFunction
embedding_fuction = MyEmbeddingFunction()


In [ ]:
redis_host = os.getenv("REDIS_HOST", "localhost")
redis_port = int(os.getenv("REDIS_PORT", 6379))

if redis_host is None or redis_port is None:
    raise ValueError("Redis host and port must be set in the environment variables.")

In [ ]:
# Need to move the respone to redis with ttl policy
import redis

redis_client = redis.Redis(
    host=redis_host,
    port=redis_port,
    decode_responses=True,
)

print(redis_client.ping())

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./ChromaDB")

In [ ]:
if "my_collection" in [collection.name for collection in client.list_collections()]:
    client.delete_collection("my_collection")
collection = client.create_collection(
    "my_collection",
    embedding_function=embedding_fuction,
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
    )

In [ ]:
open_router_api_key = os.getenv("OPEN_ROUTER_API_KEY")

if open_router_api_key is None:
    raise ValueError("OPEN_ROUTER_API_KEY environment variable is not set.")
else:
    print("OPEN_ROUTER_API_KEY environment variable is set.")

In [ ]:
#call llm
from openai import OpenAI
open_router = OpenAI(api_key=open_router_api_key, base_url="https://openrouter.ai/api/v1")

In [ ]:
SYSTEM_PROMPT = "your are an helpful assistant"

In [ ]:
#USER_PROMPT = "Explain about the python programming language"
#USER_PROMPT = "Explain about interpreted language"
USER_PROMPT = "Tell me about the python programming language"
messages = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}]

In [ ]:
def build_cache_item(user_prompt: str, embedding_id, model_name: str):
    cacheId = "cache-" + str(embedding_id)
    document = user_prompt
    metadata = {
        "timestamp": str(datetime.now()),
        "model_name": model_name,
    }
    return cacheId, document, metadata

In [ ]:
def call_llm_and_cache(messages, redis_client, collection):
    # Call the OpenRouter LLM
    response = open_router.chat.completions.create(
        model="openrouter/free",
        messages=messages,
        max_tokens=500,
        temperature=0.6
    )
    response_text = response.choices[0].message.content
    print("LLM Response:", response_text)

    if(response_text is None or response_text.strip() == ""):
        raise ValueError("LLM response is empty.")

    embedding_id = uuid.uuid4()

    cacheId, document, metadata = build_cache_item(USER_PROMPT, embedding_id, "openrouter/free")

    collection.add(
        ids=[cacheId],
        documents=[document],
        metadatas=[metadata]
    )

    redis_key = "LLM-Response:" + cacheId
    redis_client.setex(
        redis_key,
        3600,
        response_text
    )

In [ ]:
result = collection.query(
    query_texts=[USER_PROMPT],
    n_results=3
)

print("Query:", USER_PROMPT)

print("Query Result:", result)
print(len(result["ids"][0]))

cacheId = None

if len(result["ids"][0]) > 0:
    for cache_id, distance in zip(
        result["ids"][0],
        result["distances"][0]
    ):
        print("distance:", distance)

        if distance < 0.2:
            cacheId = cache_id

            redis_key = "LLM-Response:" + cacheId

            response_text = redis_client.get(redis_key)

            if response_text is not None:
                print("Cached Response:", response_text)
            else:
                print("Redis cache expired. Calling LLM.")
                call_llm_and_cache(
                    messages,
                    redis_client,
                    collection
                )

            break

    else:
        print("Not semantically similar")
        print("Calling LLM")

        call_llm_and_cache(
            messages,
            redis_client,
            collection
        )


    
    

In [ ]:
# Need to check the callings properly in all the cells